In [1]:
import requests
import pandas as pd
from requests.auth import HTTPBasicAuth
from datetime import date, timedelta

USERNAME = "gabriel.bezhi@stud.hslu.ch"
PASSWORD = "uBFh6fsx"
BASE_URL = "https://www.zefix.admin.ch/ZefixPublicREST"

auth = HTTPBasicAuth(USERNAME, PASSWORD)
headers = {"Accept": "application/json", "Content-Type": "application/json"}

In [2]:

# Zieldatum: gestern (oder manuell setzen: date(2026, 5, 15))
target_date = date.today() - timedelta(days=2)

resp = requests.get(
    f"{BASE_URL}/api/v1/sogc/bydate/{target_date.isoformat()}",
    auth=auth, headers=headers, timeout=30,
)
resp.raise_for_status()
sogc_results = resp.json()

# Endpoint liefert CH-weit → auf Kanton LU filtern
hits = sogc_results if isinstance(sogc_results, list) else sogc_results.get("list", [])
df_delta = pd.json_normalize(hits)
df_delta.head(n=7)

,sogcPublication.sogcDate,sogcPublication.sogcId,sogcPublication.registryOfCommerceId,sogcPublication.registryOfCommerceCanton,sogcPublication.registryOfCommerceJournalId,sogcPublication.registryOfCommerceJournalDate,sogcPublication.message,sogcPublication.mutationTypes,companyShort.name,companyShort.ehraid,...,companyShort.legalForm.name.fr,companyShort.legalForm.name.it,companyShort.legalForm.name.en,companyShort.legalForm.shortName.de,companyShort.legalForm.shortName.fr,companyShort.legalForm.shortName.it,companyShort.legalForm.shortName.en,companyShort.status,companyShort.sogcDate,companyShort.deletionDate
0,2026-05-15,1006651218,36,BE,9057,2026-05-11,"<FT TYPE=""F"">1899 Architekten AG</FT>, in <FT ...","[{'id': 1, 'key': 'status'}, {'id': 3, 'key': ...",1899 Architekten AG in Liquidation,856774,...,Société anonyme,Società anonima,Corporation,AG,SA,SA,Ltd,BEING_CANCELLED,2026-05-15,None
1,2026-05-15,1006651462,241,SO,2725,2026-05-11,"<FT TYPE=""F"">1&amp;One GmbH in Liquidation</FT...","[{'id': 1, 'key': 'status'}, {'id': 6, 'key': ...",1&One GmbH in Liquidation,1580818,...,Société à responsabilité limitée,Società a garanzia limitata,Limited Liability Company,GmbH,Sàrl,Sagl,LLC,BEING_CANCELLED,2026-05-15,None
2,2026-05-15,1006651045,20,ZH,22923,2026-05-11,"<FT TYPE=""F"">2A Isolationen GmbH in Liquidatio...","[{'id': 1, 'key': 'status'}, {'id': 6, 'key': ...",2A Isolationen GmbH in Liquidation,1613542,...,Société à responsabilité limitée,Società a garanzia limitata,Limited Liability Company,GmbH,Sàrl,Sagl,LLC,BEING_CANCELLED,2026-05-15,None
3,2026-05-15,1006651046,20,ZH,22924,2026-05-11,"<FT TYPE=""F"">3D360.ch Videoproduction GmbH</FT...","[{'id': 16, 'key': 'adressaenderung'}]",3D360.ch Videoproduction GmbH,1236348,...,Société à responsabilité limitée,Società a garanzia limitata,Limited Liability Company,GmbH,Sàrl,Sagl,LLC,ACTIVE,2026-05-15,None
4,2026-05-15,1006651339,130,SZ,3193,2026-05-11,"<FT TYPE=""F"">4ME Group GmbH</FT>, in <FT TYPE=...","[{'id': 11, 'key': 'zweckaenderung'}]",4ME Group GmbH,854777,...,Société à responsabilité limitée,Società a garanzia limitata,Limited Liability Company,GmbH,Sàrl,Sagl,LLC,ACTIVE,2026-05-15,None
5,2026-05-15,1006651208,36,BE,9046,2026-05-11,"<FT TYPE=""N"">4ME Group GmbH</FT>, in <FT TYPE=...","[{'id': 1, 'key': 'status'}, {'id': 2, 'key': ...",4ME Group GmbH,1749499,...,Succursale,Succursale,Branch,ZN,Succ,Succ,Bran,ACTIVE,2026-05-15,None
6,2026-05-15,1006651332,130,SZ,3186,2026-05-11,"<FT TYPE=""N"">7fridays Management und Service G...","[{'id': 1, 'key': 'status'}, {'id': 2, 'key': ...",7fridays Management und Service GmbH,1749432,...,Société à responsabilité limitée,Società a garanzia limitata,Limited Liability Company,GmbH,Sàrl,Sagl,LLC,ACTIVE,2026-05-15,None


In [3]:
def is_new_entry(mt_list):
    if not isinstance(mt_list, list):
        return False
    return any(m.get("key") == "status.neu" for m in mt_list)

df_new_lu = df_delta[
    df_delta["sogcPublication.mutationTypes"].apply(is_new_entry)
    & (df_delta["sogcPublication.registryOfCommerceCanton"] == "LU")
].reset_index(drop=True)

In [4]:
df_new_lu.head()

,sogcPublication.sogcDate,sogcPublication.sogcId,sogcPublication.registryOfCommerceId,sogcPublication.registryOfCommerceCanton,sogcPublication.registryOfCommerceJournalId,sogcPublication.registryOfCommerceJournalDate,sogcPublication.message,sogcPublication.mutationTypes,companyShort.name,companyShort.ehraid,...,companyShort.legalForm.name.fr,companyShort.legalForm.name.it,companyShort.legalForm.name.en,companyShort.legalForm.shortName.de,companyShort.legalForm.shortName.fr,companyShort.legalForm.shortName.it,companyShort.legalForm.shortName.en,companyShort.status,companyShort.sogcDate,companyShort.deletionDate
0,2026-05-15,1006651271,100,LU,4616,2026-05-11,"<FT TYPE=""N"">Ablauf-Profi Fenk</FT>, in <FT TY...","[{'id': 1, 'key': 'status'}, {'id': 2, 'key': ...",Ablauf-Profi Fenk,1749365,...,Entreprise individuelle,Ditta individuale,Sole proprietorship,EIU,EI,IPI,SP,ACTIVE,2026-05-15,None
1,2026-05-15,1006651272,100,LU,4617,2026-05-11,"<FT TYPE=""N"">BLG GLOBAL AG</FT>, in <FT TYPE=""...","[{'id': 1, 'key': 'status'}, {'id': 2, 'key': ...",BLG GLOBAL AG,1749366,...,Société anonyme,Società anonima,Corporation,AG,SA,SA,Ltd,ACTIVE,2026-05-15,None
2,2026-05-15,1006651273,100,LU,4618,2026-05-11,"<FT TYPE=""N"">Büchli Fassaden GmbH</FT>, in <FT...","[{'id': 1, 'key': 'status'}, {'id': 2, 'key': ...",Büchli Fassaden GmbH,1749367,...,Société à responsabilité limitée,Società a garanzia limitata,Limited Liability Company,GmbH,Sàrl,Sagl,LLC,ACTIVE,2026-05-15,None
3,2026-05-15,1006651274,100,LU,4619,2026-05-11,"<FT TYPE=""N"">Dairy Group GmbH</FT>, in <FT TYP...","[{'id': 1, 'key': 'status'}, {'id': 2, 'key': ...",Dairy Group GmbH,1749368,...,Société à responsabilité limitée,Società a garanzia limitata,Limited Liability Company,GmbH,Sàrl,Sagl,LLC,ACTIVE,2026-05-15,None
4,2026-05-15,1006651275,100,LU,4620,2026-05-11,"<FT TYPE=""N"">ED CONSTRUCTION &amp; TRADE AG</F...","[{'id': 1, 'key': 'status'}, {'id': 2, 'key': ...",ED CONSTRUCTION & TRADE AG,1749369,...,Société anonyme,Società anonima,Corporation,AG,SA,SA,Ltd,ACTIVE,2026-05-15,None


In [5]:
df_new_lu.columns

Index(['sogcPublication.sogcDate', 'sogcPublication.sogcId',
       'sogcPublication.registryOfCommerceId',
       'sogcPublication.registryOfCommerceCanton',
       'sogcPublication.registryOfCommerceJournalId',
       'sogcPublication.registryOfCommerceJournalDate',
       'sogcPublication.message', 'sogcPublication.mutationTypes',
       'companyShort.name', 'companyShort.ehraid', 'companyShort.uid',
       'companyShort.chid', 'companyShort.legalSeatId',
       'companyShort.legalSeat', 'companyShort.registryOfCommerceId',
       'companyShort.legalForm.id', 'companyShort.legalForm.uid',
       'companyShort.legalForm.name.de', 'companyShort.legalForm.name.fr',
       'companyShort.legalForm.name.it', 'companyShort.legalForm.name.en',
       'companyShort.legalForm.shortName.de',
       'companyShort.legalForm.shortName.fr',
       'companyShort.legalForm.shortName.it',
       'companyShort.legalForm.shortName.en', 'companyShort.status',
       'companyShort.sogcDate', 'companyShor

In [6]:
df_new_lu['companyShort.uid']

0     CHE319270603
1     CHE144292498
2     CHE378827518
3     CHE380109094
4     CHE483384987
5     CHE275688243
6     CHE217329461
7     CHE325288155
8     CHE396172862
9     CHE482398923
10    CHE232829021
11    CHE379443161
12    CHE479249406
13    CHE298395823
14    CHE195244021
15    CHE218188140
Name: companyShort.uid, dtype: object

In [8]:

import time
import requests
import pandas as pd

# BASE_URL, auth, headers werden als bereits definiert angenommen
CANTON = "LU"
SLEEP_BETWEEN = 0.3

# 1-stellige Präfixe: a-z + 0-9 = 36 Kombinationen
prefixes = df_new_lu['companyShort.uid']

all_hits = []

for i, prefix in enumerate(prefixes, 1):
    payload = {
        "name": f"{prefix}",
        "canton": CANTON,
        "activeOnly": True,
    }

    try:
        resp = requests.post(
            f"{BASE_URL}/api/v1/company/search",
            auth=auth, headers=headers, json=payload, timeout=30,
        )
        resp.raise_for_status()
        search_results = resp.json()
        hits = search_results if isinstance(search_results, list) else search_results.get("list", [])
    except Exception as e:
        print(f"[{i}/{len(prefixes)}] {prefix}*: FEHLER {e}")
        continue

    if hits:
        all_hits.extend(hits)
        print(f"[{i}/{len(prefixes)}] {prefix}*: +{len(hits)} (total: {len(all_hits)})")
    else:
        print(f"[{i}/{len(prefixes)}] {prefix}*: 0 Treffer")

    time.sleep(SLEEP_BETWEEN)

# In DataFrame packen und über uid deduplizieren
df_search = pd.json_normalize(all_hits)
if "uid" in df_search.columns:
    before = len(df_search)
    df_search = df_search.drop_duplicates(subset="uid", keep="first").reset_index(drop=True)
    print(f"\nDedup: {before} → {len(df_search)} unique Firmen")

print(f"Fertig: {len(df_search)} unique Firmen in Kanton {CANTON}")
df_search

[1/16] CHE319270603*: +1 (total: 1)
[2/16] CHE144292498*: +1 (total: 2)
[3/16] CHE378827518*: +1 (total: 3)
[4/16] CHE380109094*: +1 (total: 4)
[5/16] CHE483384987*: +1 (total: 5)
[6/16] CHE275688243*: +1 (total: 6)
[7/16] CHE217329461*: +1 (total: 7)
[8/16] CHE325288155*: +1 (total: 8)
[9/16] CHE396172862*: +1 (total: 9)
[10/16] CHE482398923*: +1 (total: 10)
[11/16] CHE232829021*: +1 (total: 11)
[12/16] CHE379443161*: +1 (total: 12)
[13/16] CHE479249406*: +1 (total: 13)
[14/16] CHE298395823*: +1 (total: 14)
[15/16] CHE195244021*: +1 (total: 15)
[16/16] CHE218188140*: +1 (total: 16)

Dedup: 16 → 16 unique Firmen
Fertig: 16 unique Firmen in Kanton LU


,name,ehraid,uid,chid,legalSeatId,legalSeat,registryOfCommerceId,status,sogcDate,deletionDate,legalForm.id,legalForm.uid,legalForm.name.de,legalForm.name.fr,legalForm.name.it,legalForm.name.en,legalForm.shortName.de,legalForm.shortName.fr,legalForm.shortName.it,legalForm.shortName.en
0,Ablauf-Profi Fenk,1749365,CHE319270603,CH10018236768,1008,Schüpfheim,100,ACTIVE,2026-05-15,None,1,0101,Einzelunternehmen,Entreprise individuelle,Ditta individuale,Sole proprietorship,EIU,EI,IPI,SP
1,BLG GLOBAL AG,1749366,CHE144292498,CH10038236704,1061,Luzern,100,ACTIVE,2026-05-15,None,3,0106,Aktiengesellschaft,Société anonyme,Società anonima,Corporation,AG,SA,SA,Ltd
2,Büchli Fassaden GmbH,1749367,CHE378827518,CH10048236745,1061,Luzern,100,ACTIVE,2026-05-15,None,4,0107,Gesellschaft mit beschränkter Haftung,Société à responsabilité limitée,Società a garanzia limitata,Limited Liability Company,GmbH,Sàrl,Sagl,LLC
3,Dairy Group GmbH,1749368,CHE380109094,CH10048235678,1140,Reiden,100,ACTIVE,2026-05-15,None,4,0107,Gesellschaft mit beschränkter Haftung,Société à responsabilité limitée,Società a garanzia limitata,Limited Liability Company,GmbH,Sàrl,Sagl,LLC
4,ED CONSTRUCTION & TRADE AG,1749369,CHE483384987,CH10038236728,1063,Meggen,100,ACTIVE,2026-05-15,None,3,0106,Aktiengesellschaft,Société anonyme,Società anonima,Corporation,AG,SA,SA,Ltd
5,ek dental GmbH,1749371,CHE275688243,CH10048236776,1084,Eich,100,ACTIVE,2026-05-15,None,4,0107,Gesellschaft mit beschränkter Haftung,Société à responsabilité limitée,Società a garanzia limitata,Limited Liability Company,GmbH,Sàrl,Sagl,LLC
6,Hans Schmid Stiftung,1749372,CHE217329461,CH10078236751,1052,Buchrain,100,ACTIVE,2026-05-15,None,7,0110,Stiftung,Fondation,Fondazione,Foundation,Stift,Fond,Fond,Found
7,Immo Friendship gmbh,1749373,CHE325288155,CH10048236713,1098,Ruswil,100,ACTIVE,2026-05-15,None,4,0107,Gesellschaft mit beschränkter Haftung,Société à responsabilité limitée,Società a garanzia limitata,Limited Liability Company,GmbH,Sàrl,Sagl,LLC
8,Life Athlete GmbH,1749375,CHE396172862,CH10048236784,1061,Luzern,100,ACTIVE,2026-05-15,None,4,0107,Gesellschaft mit beschränkter Haftung,Société à responsabilité limitée,Società a garanzia limitata,Limited Liability Company,GmbH,Sàrl,Sagl,LLC
9,MAY STUDIO LÊ,1749378,CHE482398923,CH10018236264,1024,Emmen,100,ACTIVE,2026-05-15,None,1,0101,Einzelunternehmen,Entreprise individuelle,Ditta individuale,Sole proprietorship,EIU,EI,IPI,SP


In [9]:
def fetch_details(uid: str) -> list[dict]:
    """Holt alle Detail-Einträge zu einer UID. Gibt leere Liste bei Fehler zurück."""
    try:
        r = requests.get(
            f"{BASE_URL}/api/v1/company/uid/{uid}",
            auth=auth, headers={"Accept": "application/json"}, timeout=30,
        )
        r.raise_for_status()
        data = r.json()
        return data if isinstance(data, list) else [data]
    except requests.HTTPError as e:
        print(f"  ! Fehler bei UID {uid}: {e}")
        return []

In [10]:
all_details = []
for uid in df_search["uid"].dropna().unique():
    print(f"Lade Details für {uid} ...")
    all_details.extend(fetch_details(uid))
    time.sleep(0.2)

df_details = pd.json_normalize(all_details, sep=".")

Lade Details für CHE319270603 ...
Lade Details für CHE144292498 ...
Lade Details für CHE378827518 ...
Lade Details für CHE380109094 ...
Lade Details für CHE483384987 ...
Lade Details für CHE275688243 ...
Lade Details für CHE217329461 ...
Lade Details für CHE325288155 ...
Lade Details für CHE396172862 ...
Lade Details für CHE482398923 ...
Lade Details für CHE232829021 ...
Lade Details für CHE379443161 ...
Lade Details für CHE479249406 ...
Lade Details für CHE298395823 ...
Lade Details für CHE195244021 ...
Lade Details für CHE218188140 ...


In [13]:
df_merged = df_search.merge(df_details, on="uid", how="left", suffixes=("_search", "_detail"))

print(f"\nSuche: {len(df_search)} Treffer")
print(f"Details: {len(df_details)} Einträge")
print(f"Merged: {len(df_merged)} Zeilen")

df_merged.head(n=10)


Suche: 16 Treffer
Details: 16 Einträge
Merged: 16 Zeilen


,name_search,ehraid_search,uid,chid_search,legalSeatId_search,legalSeat_search,registryOfCommerceId_search,status_search,sogcDate_search,deletionDate_search,...,address.street,address.houseNumber,address.addon,address.poBox,address.city,address.swissZipCode,zefixDetailWeb.de,zefixDetailWeb.fr,zefixDetailWeb.it,zefixDetailWeb.en
0,Ablauf-Profi Fenk,1749365,CHE319270603,CH10018236768,1008,Schüpfheim,100,ACTIVE,2026-05-15,None,...,Fontanne,8,,,Schüpfheim,6170,https://www.zefix.admin.ch/de/search/entity/li...,https://www.zefix.admin.ch/fr/search/entity/li...,https://www.zefix.admin.ch/it/search/entity/li...,https://www.zefix.admin.ch/en/search/entity/li...
1,BLG GLOBAL AG,1749366,CHE144292498,CH10038236704,1061,Luzern,100,ACTIVE,2026-05-15,None,...,Zentralstrasse,44,,,Luzern,6003,https://www.zefix.admin.ch/de/search/entity/li...,https://www.zefix.admin.ch/fr/search/entity/li...,https://www.zefix.admin.ch/it/search/entity/li...,https://www.zefix.admin.ch/en/search/entity/li...
2,Büchli Fassaden GmbH,1749367,CHE378827518,CH10048236745,1061,Luzern,100,ACTIVE,2026-05-15,None,...,Reusszopf,21,,,Luzern,6015,https://www.zefix.admin.ch/de/search/entity/li...,https://www.zefix.admin.ch/fr/search/entity/li...,https://www.zefix.admin.ch/it/search/entity/li...,https://www.zefix.admin.ch/en/search/entity/li...
3,Dairy Group GmbH,1749368,CHE380109094,CH10048235678,1140,Reiden,100,ACTIVE,2026-05-15,None,...,Bodenachermatte,8,,,Reiden,6260,https://www.zefix.admin.ch/de/search/entity/li...,https://www.zefix.admin.ch/fr/search/entity/li...,https://www.zefix.admin.ch/it/search/entity/li...,https://www.zefix.admin.ch/en/search/entity/li...
4,ED CONSTRUCTION & TRADE AG,1749369,CHE483384987,CH10038236728,1063,Meggen,100,ACTIVE,2026-05-15,None,...,Huobmattstrasse,3,,,Meggen,6045,https://www.zefix.admin.ch/de/search/entity/li...,https://www.zefix.admin.ch/fr/search/entity/li...,https://www.zefix.admin.ch/it/search/entity/li...,https://www.zefix.admin.ch/en/search/entity/li...
5,ek dental GmbH,1749371,CHE275688243,CH10048236776,1084,Eich,100,ACTIVE,2026-05-15,None,...,Ibrigweidstrasse,3b,,,Eich,6205,https://www.zefix.admin.ch/de/search/entity/li...,https://www.zefix.admin.ch/fr/search/entity/li...,https://www.zefix.admin.ch/it/search/entity/li...,https://www.zefix.admin.ch/en/search/entity/li...
6,Hans Schmid Stiftung,1749372,CHE217329461,CH10078236751,1052,Buchrain,100,ACTIVE,2026-05-15,None,...,Neuhaltenring,1,,,Ebikon,6030,https://www.zefix.admin.ch/de/search/entity/li...,https://www.zefix.admin.ch/fr/search/entity/li...,https://www.zefix.admin.ch/it/search/entity/li...,https://www.zefix.admin.ch/en/search/entity/li...
7,Immo Friendship gmbh,1749373,CHE325288155,CH10048236713,1098,Ruswil,100,ACTIVE,2026-05-15,None,...,Grindel,1,,,Ruswil,6017,https://www.zefix.admin.ch/de/search/entity/li...,https://www.zefix.admin.ch/fr/search/entity/li...,https://www.zefix.admin.ch/it/search/entity/li...,https://www.zefix.admin.ch/en/search/entity/li...
8,Life Athlete GmbH,1749375,CHE396172862,CH10048236784,1061,Luzern,100,ACTIVE,2026-05-15,None,...,Neustadtstrasse,6,,,Luzern,6003,https://www.zefix.admin.ch/de/search/entity/li...,https://www.zefix.admin.ch/fr/search/entity/li...,https://www.zefix.admin.ch/it/search/entity/li...,https://www.zefix.admin.ch/en/search/entity/li...
9,MAY STUDIO LÊ,1749378,CHE482398923,CH10018236264,1024,Emmen,100,ACTIVE,2026-05-15,None,...,Gerliswilstrasse,32,,,Emmenbrücke,6020,https://www.zefix.admin.ch/de/search/entity/li...,https://www.zefix.admin.ch/fr/search/entity/li...,https://www.zefix.admin.ch/it/search/entity/li...,https://www.zefix.admin.ch/en/search/entity/li...


In [14]:
import re
import time
import requests
from bs4 import BeautifulSoup
from datetime import datetime, date
from urllib.parse import urlparse

HEADERS_HRK = {
    "User-Agent": (
        "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/120.0 Safari/537.36"
    ),
    "Accept-Language": "de-CH,de;q=0.9",
}

def _parse_eintragsdatum(response_text: str) -> date | None:
    m = re.search(
        r"Eingetragen am</span>\s*<span>(\d{2}\.\d{2}\.\d{4})</span>",
        response_text,
    )
    if not m:
        soup = BeautifulSoup(response_text, "html.parser")
        text = soup.get_text(" ", strip=True)
        m = re.search(
            r"Eingetragen\s+am[\s:]*?(\d{2}\.\d{2}\.\d{4})",
            text,
            flags=re.IGNORECASE,
        )
        if not m:
            return None
    return datetime.strptime(m.group(1), "%d.%m.%Y").date()

def scrape_eintragsdatum_from_url(
    url: str,
    session: requests.Session | None = None,
    timeout: int = 30,
) -> date | None:
    if not isinstance(url, str) or not url:
        return None
    
    sess = session or requests.Session()
    
    # 1) Seite laden → ViewState einsammeln
    r1 = sess.get(url, headers=HEADERS_HRK, timeout=timeout)
    r1.raise_for_status()
    soup = BeautifulSoup(r1.text, "html.parser")
    vs_el = soup.find("input", {"name": "javax.faces.ViewState"})
    if not vs_el:
        return None
    view_state = vs_el.get("value")
    
    # Origin aus der URL ableiten (für den Referer/Origin-Header)
    parsed = urlparse(url)
    origin = f"{parsed.scheme}://{parsed.hostname}"
    
    # 2) AJAX-Postback fürs Lazy-Panel
    ajax_headers = {
        **HEADERS_HRK,
        "Faces-Request": "partial/ajax",
        "X-Requested-With": "XMLHttpRequest",
        "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
        "Origin": origin,
        "Referer": r1.url,
    }
    data = {
        "javax.faces.partial.ajax": "true",
        "javax.faces.source": "idAuszugForm:auszugContentPanel",
        "javax.faces.partial.execute": "idAuszugForm:auszugContentPanel",
        "javax.faces.partial.render": "idAuszugForm:auszugContentPanel",
        "idAuszugForm:auszugContentPanel_load": "true",
        "idAuszugForm": "idAuszugForm",
        "javax.faces.ViewState": view_state,
    }
    r2 = sess.post(url, data=data, headers=ajax_headers, timeout=timeout)
    r2.raise_for_status()
    
    return _parse_eintragsdatum(r2.text)

In [15]:
from tqdm import tqdm

tqdm.pandas()
sess = requests.Session()

def safe_scrape(url):
    try:
        result = scrape_eintragsdatum_from_url(url, session=sess)
        time.sleep(0.2)
        return result
    except Exception as e:
        print(f"Fehler bei {url}: {e}")
        return None

df_merged["eintragsdatum"] = df_merged["cantonalExcerptWeb"].progress_apply(safe_scrape)

100%|███████████████████████████████████████████| 16/16 [00:09<00:00,  1.62it/s]


In [16]:
df_merged.head()

,name_search,ehraid_search,uid,chid_search,legalSeatId_search,legalSeat_search,registryOfCommerceId_search,status_search,sogcDate_search,deletionDate_search,...,address.houseNumber,address.addon,address.poBox,address.city,address.swissZipCode,zefixDetailWeb.de,zefixDetailWeb.fr,zefixDetailWeb.it,zefixDetailWeb.en,eintragsdatum
0,Ablauf-Profi Fenk,1749365,CHE319270603,CH10018236768,1008,Schüpfheim,100,ACTIVE,2026-05-15,None,...,8,,,Schüpfheim,6170,https://www.zefix.admin.ch/de/search/entity/li...,https://www.zefix.admin.ch/fr/search/entity/li...,https://www.zefix.admin.ch/it/search/entity/li...,https://www.zefix.admin.ch/en/search/entity/li...,2026-05-11
1,BLG GLOBAL AG,1749366,CHE144292498,CH10038236704,1061,Luzern,100,ACTIVE,2026-05-15,None,...,44,,,Luzern,6003,https://www.zefix.admin.ch/de/search/entity/li...,https://www.zefix.admin.ch/fr/search/entity/li...,https://www.zefix.admin.ch/it/search/entity/li...,https://www.zefix.admin.ch/en/search/entity/li...,2026-05-11
2,Büchli Fassaden GmbH,1749367,CHE378827518,CH10048236745,1061,Luzern,100,ACTIVE,2026-05-15,None,...,21,,,Luzern,6015,https://www.zefix.admin.ch/de/search/entity/li...,https://www.zefix.admin.ch/fr/search/entity/li...,https://www.zefix.admin.ch/it/search/entity/li...,https://www.zefix.admin.ch/en/search/entity/li...,2026-05-11
3,Dairy Group GmbH,1749368,CHE380109094,CH10048235678,1140,Reiden,100,ACTIVE,2026-05-15,None,...,8,,,Reiden,6260,https://www.zefix.admin.ch/de/search/entity/li...,https://www.zefix.admin.ch/fr/search/entity/li...,https://www.zefix.admin.ch/it/search/entity/li...,https://www.zefix.admin.ch/en/search/entity/li...,2026-05-11
4,ED CONSTRUCTION & TRADE AG,1749369,CHE483384987,CH10038236728,1063,Meggen,100,ACTIVE,2026-05-15,None,...,3,,,Meggen,6045,https://www.zefix.admin.ch/de/search/entity/li...,https://www.zefix.admin.ch/fr/search/entity/li...,https://www.zefix.admin.ch/it/search/entity/li...,https://www.zefix.admin.ch/en/search/entity/li...,2026-05-11
